In [ ]:
UPDATE=False

In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections
import matplotlib.pyplot as plt
# %matplotlib inline

import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

# SENTIMENT

In [ ]:
if (UPDATE):
    df_sentiment = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTime.pkl")
    df_sentiment.index = pd.to_datetime(df_sentiment.index)
    df_sentiment.Date = pd.to_datetime(df_sentiment.Date)
    df_sentiment.sort_values(by="Time",inplace=True)
    df_sentiment.index = df_sentiment.Time.to_period(freq='T').index
    df_sentiment["sentiment"] = df_sentiment["sentiment"]/df_sentiment["MarketCap"]
    df_sentiment.index = df_sentiment.Time
    df_sentiment.Date = df_sentiment.index
    df_sentiment.rename(columns={
                    "sentiment":"Sentiment",
                    "Sentiment_ln_return":"Sentiment return",
                    "Sentiment_ln_return(t-1)":"Sentiment return(t-1)",
                    "pos":"Positive",
                    "pos(t-1)":"Positive(t-1)",
                    "pos(t-2)":"Positive(t-2)",
                    "pos_ln_return":"Positive return",
                    "pos_ln_return(t-1)":"Positive return(t-1)",
                    "neg":"Negative",
                    "neg(t-1)":"Negative(t-1)", 
                    "neg(t-2)":"Negative(t-2)",
                    "neg_ln_return":"Negative return", 
                    "neg_ln_return(t-1)":"Negative return(t-1)", 
                    "polarity":"Polarity",
                    "polarity(t-1)":"Polarity(t-1)",
                    "polarityB":"PolarityB",
                    "subjectivity":"Subjectivity",
                    "subjectivity(t-1)":"Subjectivity(t-1)",
                    "PressReleaseCount_ln_return":"PressReleaseCount return",
                    "PressReleaseCount_ln_return(t-1)":"PressReleaseCount return(t-1)",
                    "WordCount_ln_return":"WordCount return",
                    "WordCount_ln_return(t-1)":"WordCount return(t-1)",
                    "day":"Day of Week",
                    "month":"Month",
                    "month-year":"Month-Year",
                    "quarter":"Quarter",
                    "year":"Year",
                    "GI_Sentiment_ln_return":"GI_Sentiment return",
                    "GI_Negative_ln_return":"GI_Negative return",
                    "GI_Positive_ln_return":"GI_Positive return"}
                    ,inplace=True)
    df_sentiment = df_sentiment.ffill()

    df_sentiment["Sentiment(t-1)"] = df_sentiment["Sentiment"].shift(periods=1)
    df_sentiment["Sentiment(t-2)"] = df_sentiment["Sentiment"].shift(periods=2)
    df_sentiment["Sentiment(t-1)"] = df_sentiment["Sentiment(t-1)"].ffill()
    df_sentiment["Sentiment(t-2)"] = df_sentiment["Sentiment(t-2)"].ffill()
    # SENTIMENT RETURN
    # df_sentiment["Sentiment_return"]=(df_sentiment["Sentiment(t-1)"]-df_sentiment["Sentiment(t-2)"])/df_sentiment["Sentiment(t-2)"]
    # SENTIMENT LN RETURN
    df_sentiment["Sentiment return"]=np.log(df_sentiment["Sentiment"]/df_sentiment["Sentiment(t-1)"])
    df_sentiment["Sentiment return(t-1)"]=np.log(df_sentiment["Sentiment(t-1)"]/df_sentiment["Sentiment(t-2)"])
    # SENTIMENT EWM
    df_sentiment["Sentiment_ewm10"]=df_sentiment["Sentiment"].ewm(10).mean()
    df_sentiment["Sentiment_ewm20"]=df_sentiment["Sentiment"].ewm(20).mean()
    # POSITIVE LN RETURN
    df_sentiment["Positive(t-1)"] = df_sentiment["Positive"].shift(periods=1)
    df_sentiment["Positive(t-2)"] = df_sentiment["Positive"].shift(periods=2)
    df_sentiment["Positive(t-1)"] = df_sentiment["Positive(t-1)"].ffill()
    df_sentiment["Positive(t-2)"] = df_sentiment["Positive(t-2)"].ffill()
    df_sentiment["Positive return"]=np.log(df_sentiment["Positive"]/df_sentiment["Positive(t-1)"])
    df_sentiment["Positive return(t-1)"]=np.log(df_sentiment["Positive(t-1)"]/df_sentiment["Positive(t-2)"])
    # NEGATIVE LN RETURN
    df_sentiment["Negative(t-1)"] = df_sentiment["Negative"].shift(periods=1)
    df_sentiment["Negative(t-2)"] = df_sentiment["Negative"].shift(periods=2)
    df_sentiment["Negative(t-1)"] = df_sentiment["Negative(t-1)"].ffill()
    df_sentiment["Negative(t-2)"] = df_sentiment["Negative(t-2)"].ffill()
    df_sentiment["Negative return"]=np.log(df_sentiment["Negative"]/df_sentiment["Negative(t-1)"])
    df_sentiment["Negative return(t-1)"]=np.log(df_sentiment["Negative(t-1)"]/df_sentiment["Negative(t-2)"])
    df_sentiment["Polarity(t-1)"] = df_sentiment["Polarity"].shift(periods=1)
    df_sentiment["Subjectivity(t-1)"] = df_sentiment["Subjectivity"].shift(periods=1)
    df_sentiment['WordCount'] = df_sentiment['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text
    df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount"].shift(periods=1)
    df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount"].shift(periods=2)
    df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount(t-1)"].ffill()
    df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount(t-2)"].ffill()
    df_sentiment["WordCount return"]=np.log(df_sentiment["WordCount"]/df_sentiment["WordCount(t-1)"])
    df_sentiment.index = df_sentiment.Date.to_period(freq='T').index
    df_sentiment.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentiment-1m.pkl")
else:
    df_sentiment = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentiment-1m.pkl")

In [ ]:
df_sentiment

In [ ]:
df_sentiment.index.unique()

# VIX

In [ ]:
import datetime, pytz
from time import timezone

if (UPDATE):
    df_vix = pd.read_csv("/Users/Farid Srouji/Documents/Research/master/Data/VIX/vix-1m.csv",delimiter=";", decimal=",",names=["Date", "Minute", "Open", "High","Low","Close","Volume"])
    # tz_chicago = pytz.timezone('America/Chicago')
    df_vix.index = pd.to_datetime(df_vix['Date'] + ' ' + df_vix['Minute'], utc=True)\
                        .map(lambda x: x.tz_convert('America/Chicago'))#,dtype='datetime64[ns, Chi/Kolkata]')#tzinfo=tz_chicago)
    df_vix["Open"] = df_vix["Open"].astype(str).astype(float)
    df_vix["High"] = df_vix["High"].astype(str).astype(float)
    df_vix["Low"] = df_vix["Low"].astype(str).astype(float)
    df_vix["Close"] = df_vix["Close"].astype(str).astype(float)
    df_vix["Volume"] = df_vix["Volume"].astype(str).astype(float)

    df_vix = df_vix["2020-06-01":"2022-06-01"]
    df_vix.Date = df_vix.index
    df_vix.Date = pd.to_datetime(df_vix.Date)
    df_vix.index = df_vix.Date.to_period(freq='T').index
    # df_vix.index=df_vix['Date'].dt.strftime('%Y-%m-%d %H')
    df_vix.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix-1m.pkl")
else:
    df_vix = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix-1m.pkl")

In [ ]:
sentiment = df_sentiment.Time.dt.time.keys().to_list()

In [ ]:
vix = df_vix.index.to_list()

In [ ]:
sentiment = set(sentiment)
vix = set(vix)

In [ ]:
len(sentiment)

In [ ]:
len(vix)

In [ ]:
len(vix.intersection(sentiment))

In [ ]:
len(df_vix.index.unique())

In [ ]:
df_vix.drop(columns={"Open","High","Low"},inplace=True)
df_vix.rename(columns={"Close":"VIX"},inplace=True)
# df_vix.index = pd.to_datetime(df_vix.index)
# df_vix["Date"]= pd.to_datetime(df_vix.index)
df_vix = df_vix.loc[df_vix["Date"]>"2020-05-31"]

df_vix["VIX(t-1)"] = df_vix["VIX"].shift(periods=1)
df_vix["VIX(t-1)"].ffill(inplace=True)

df_vix["vix_ln_return"]=np.log(df_vix["VIX"]/df_vix["VIX(t-1)"])
df_vix["vix_ln_return"].ffill(inplace=True)

In [ ]:
df_vix.drop(columns={"Date"},inplace=True)

In [ ]:
df = df_sentiment.merge(df_vix,right_index=True, left_index=True)
df = df.ffill()
df = df.sort_index()

In [ ]:
df.dropna(inplace=True)

mx = df.loc[df['Sentiment'] != np.inf, 'Sentiment'].max()
df['Sentiment'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Sentiment'] != -np.inf, 'Sentiment'].min()
df['Sentiment'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['Sentiment'] != np.inf, 'Sentiment'].max()
df['Sentiment'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Sentiment'] != -np.inf, 'Sentiment'].min()
df['Sentiment'].replace(-np.inf,mn,inplace=True)

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['Year'] = [d.year for d in df.Date]
df['Month'] = [d.strftime('%b') for d in df.Date]
df['Day of Week'] = [d.day_name() for d in df.Date]
years = df['Year'].unique()
df['Month-Year'] = [d.strftime('%b') + str(d.year) for d in df.Date]
df['Quarter'] = pd.PeriodIndex(df.Date, freq='Q')

mx = df.loc[df['Sentiment return'] != np.inf, 'Sentiment return'].max()
df['Sentiment return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Sentiment return'] != -np.inf, 'Sentiment return'].min()
df['Sentiment return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['vix_ln_return'] != np.inf, 'vix_ln_return'].max()
df['vix_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['vix_ln_return'] != -np.inf, 'vix_ln_return'].min()
df['vix_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['Positive return'] != np.inf, 'Positive return'].max()
df['Positive return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Positive return'] != -np.inf, 'Positive return'].min()
df['Positive return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['Negative return'] != np.inf, 'Negative return'].max()
df['Negative return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Negative return'] != -np.inf, 'Negative return'].min()
df['Negative return'].replace(-np.inf,mn,inplace=True)

In [ ]:
df.columns

In [ ]:
df.sort_index(inplace=True)

In [ ]:
# df[["VIX","Sentiment"]].plot.scatter(x = 'Sentiment', y = 'VIX', c = 'red');

In [ ]:
df.rename(columns={"vix":"VIX",
                "vix(t-1)":"VIX(t-1)",
                "vix_ln_return":"VIX return",
                "PressReleaseCount_ln_return":"PressReleaseCount return",
                "PressReleaseCount_ln_return(t-1)":"PressReleaseCount return(t-1)",
                "WordCount_ln_return":"WordCount return",
                "WordCount_ln_return(t-1)":"WordCount return(t-1)",
                "day":"Day of Week",
                "month":"Month",
                "month-year":"Month-Year",
                "quarter":"Quarter",
                "year":"Year",
                "GI_Sentiment_ln_return":"GI_Sentiment return",
                "GI_Negative_ln_return":"GI_Negative return",
                "GI_Positive_ln_return":"GI_Positive return"}
                ,inplace=True)

In [ ]:
dfMinute = df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfMinute.pkl")

In [ ]:
dfMinute = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfMinute.pkl")

In [ ]:
dfMinute